# Beyond Frequency: Knowledge Graphs for Fashion Trend Analysis

**Author:** Diego Félix Beserra de Lima  
**Institution:** UFCG — Programa de Pós-Graduação em Ciência da Computação  
**Discipline:** Reprodutibilidade em Pesquisa em Computação  
**Date:** June 2026

---

## About this Notebook

This notebook implements a comparative experiment between two conditions for answering questions about fashion trends:

- **Condition A (LLM-only):** raw text sent directly to the LLM
- **Condition B (KG+LLM):** structured knowledge graph consulted and sent to the LLM

**Research Question:** To what extent are responses about fashion trends generated via a knowledge graph more interpretable and grounded than responses generated directly from raw text?

**Pipeline:**
```
data/raw/ → [01-clean] → data/processed/
         → [02-extract-triples] → data/processed/triples/
         → [03-build-kg] → data/processed/kg/
         → [04-query] → results/
```

**AI Tool Declaration (CNPq Portaria 2.664/2026):**  
Claude (Anthropic), model claude-sonnet-4-6, was used in June 2026 to assist in script development, pattern analysis of raw data, and pipeline structuring. All content was reviewed and validated by the author, who assumes full responsibility.

---
## Step 0 — Environment Setup

In [1]:
# Install dependencies
!pip install openai networkx --quiet

In [2]:
# Clone the repository
!git clone https://github.com/diegofelixlima/beyond-frequency.git
%cd beyond-frequency

Cloning into 'beyond-frequency'...
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 106 (delta 40), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (106/106), 61.54 KiB | 7.69 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/beyond-frequency


In [3]:
# Load OpenAI API key from Colab Secrets
# To add your key: click the key icon in the left sidebar → Add new secret → Name: OPENAI_API_KEY
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("API key loaded successfully.")

API key loaded successfully.


---
## Step 1 — Clean Raw Reviews

Removes site navigation, footer, and metadata from raw Ctrl+A collected texts.  
Keeps only editorial content: title, vibe, review body, quote, and wrap up.

**Input:** `data/raw/*.txt` (immutable — never modified)  
**Output:** `data/processed/*.txt`

In [4]:
!python src/01-clean.py

Arquivos encontrados em data/raw: 9

[OK] 2025-lfw-burberry-review.txt → 568 palavras
[OK] 2025-lfw-di-petsa-review.txt → 641 palavras
[OK] 2025-lfw-dilara-findikoglu-review.txt → 773 palavras
[OK] 2025-lfw-erdem-review.txt → 828 palavras
[OK] 2025-lfw-fashion-east-review.txt → 601 palavras
[OK] 2025-lfw-jawara-alleyne-review.txt → 596 palavras
[OK] 2025-lfw-roksanda-review.txt → 489 palavras
[OK] 2025-lfw-simone-rocha-review.txt → 612 palavras
[OK] 2025-lfw-ss-daley-review.txt → 573 palavras

Concluído: 9 processados, 0 com erro.


---
## Step 2 — Extract Semantic Triples

Uses the OpenAI API (gpt-4o-mini, temperature=0) to extract semantic triples from each review.  
Triples follow the format: `{subject, relation, object}`

**Input:** `data/processed/*.txt`  
**Output:** `data/processed/triples/*.json`

**Model:** gpt-4o-mini  
**Temperature:** 0 (deterministic)

In [5]:
!python src/02-extract-triples.py

Arquivos encontrados em data/processed: 9

Processando: 2025-lfw-burberry-review.txt
[OK] 2025-lfw-burberry-review.txt → 23 triplas extraídas
Processando: 2025-lfw-di-petsa-review.txt
[OK] 2025-lfw-di-petsa-review.txt → 24 triplas extraídas
Processando: 2025-lfw-dilara-findikoglu-review.txt
[OK] 2025-lfw-dilara-findikoglu-review.txt → 24 triplas extraídas
Processando: 2025-lfw-erdem-review.txt
[OK] 2025-lfw-erdem-review.txt → 22 triplas extraídas
Processando: 2025-lfw-fashion-east-review.txt
[OK] 2025-lfw-fashion-east-review.txt → 24 triplas extraídas
Processando: 2025-lfw-jawara-alleyne-review.txt
[OK] 2025-lfw-jawara-alleyne-review.txt → 21 triplas extraídas
Processando: 2025-lfw-roksanda-review.txt
[OK] 2025-lfw-roksanda-review.txt → 21 triplas extraídas
Processando: 2025-lfw-simone-rocha-review.txt
[OK] 2025-lfw-simone-rocha-review.txt → 22 triplas extraídas
Processando: 2025-lfw-ss-daley-review.txt
[OK] 2025-lfw-ss-daley-review.txt → 21 triplas extraídas

Concluído: 9 processados,

In [6]:
# Inspect extracted triples from one review
import json

sample_file = "data/processed/triples/2025-lfw-erdem-review.json"
with open(sample_file, "r", encoding="utf-8") as f:
    triples = json.load(f)

print(f"Sample: Erdem — {len(triples)} triples extracted\n")
for t in triples[:10]:
    print(f"  {t['subject']} --[{t['relation']}]--> {t['object']}")
print(f"  ... and {len(triples)-10} more")

Sample: Erdem — 22 triples extracted

  collection --[DESIGNED_BY]--> Erdem Moralıoğlu
  collection --[INSPIRED_BY]--> Kaye Donachie
  collection --[USES_MATERIAL]--> organza
  collection --[USES_MATERIAL]--> lace
  collection --[USES_MATERIAL]--> cotton
  collection --[USES_MATERIAL]--> neoprene
  collection --[USES_COLOR]--> burgundy
  collection --[FEATURES_SILHOUETTE]--> bandeau dress
  collection --[FEATURES_SILHOUETTE]--> column dress
  collection --[FEATURES_SILHOUETTE]--> drop-waist flapper style
  ... and 12 more


In [7]:
with open("data/processed/triples/2025-lfw-erdem-review.json", "r") as f:
    triples = json.load(f)

for t in triples:
    print(f"  {t['subject']} --[{t['relation']}]--> {t['object']}")

  collection --[DESIGNED_BY]--> Erdem Moralıoğlu
  collection --[INSPIRED_BY]--> Kaye Donachie
  collection --[USES_MATERIAL]--> organza
  collection --[USES_MATERIAL]--> lace
  collection --[USES_MATERIAL]--> cotton
  collection --[USES_MATERIAL]--> neoprene
  collection --[USES_COLOR]--> burgundy
  collection --[FEATURES_SILHOUETTE]--> bandeau dress
  collection --[FEATURES_SILHOUETTE]--> column dress
  collection --[FEATURES_SILHOUETTE]--> drop-waist flapper style
  collection --[REFERENCES_CULTURE]--> domesticity
  collection --[CO_OCCURS_WITH]--> technical techniques
  collection --[CO_OCCURS_WITH]--> analogue techniques
  collection --[PRESENTED_AT]--> Fall 2025 Fashion Show
  Kaye Donachie --[REFERENCES_CULTURE]--> Scottish
  Kaye Donachie --[ASSOCIATED_WITH]--> ethereal renderings
  Kaye Donachie --[ASSOCIATED_WITH]--> poetic licence
  Kaye Donachie --[ASSOCIATED_WITH]--> emotive exploration
  collection --[INSPIRED_BY]--> artistic technique
  collection --[USES_COLOR]--> muted

---
## Step 3 — Build Knowledge Graph

Constructs a directed graph from all extracted triples using NetworkX.  
Each edge carries the relation type and source review.

**Input:** `data/processed/triples/*.json`  
**Output:** `data/processed/kg/fashion_kg.graphml`

**Note:** NetworkX is used as a simplification for reproducibility.  
The full dissertation implementation will use Neo4j.

In [8]:
!python src/03-build-kg.py

Arquivos de triplas encontrados: 9

[OK] 2025-lfw-burberry-review.json → 23 triplas adicionadas ao grafo
[OK] 2025-lfw-di-petsa-review.json → 24 triplas adicionadas ao grafo
[OK] 2025-lfw-dilara-findikoglu-review.json → 24 triplas adicionadas ao grafo
[OK] 2025-lfw-erdem-review.json → 22 triplas adicionadas ao grafo
[OK] 2025-lfw-fashion-east-review.json → 24 triplas adicionadas ao grafo
[OK] 2025-lfw-jawara-alleyne-review.json → 21 triplas adicionadas ao grafo
[OK] 2025-lfw-roksanda-review.json → 21 triplas adicionadas ao grafo
[OK] 2025-lfw-simone-rocha-review.json → 22 triplas adicionadas ao grafo
[OK] 2025-lfw-ss-daley-review.json → 21 triplas adicionadas ao grafo

Total de triplas no grafo: 202

--- Estatísticas do Grafo ---
Nós (entidades):  197
Arestas (relações): 194

Top 10 entidades mais conectadas:
  collection: 122 conexões
  burberry fall 2025 fashion show: 23 conexões
  roksanda fall 2025 fashion show: 20 conexões
  Fashion East: 8 conexões
  Nuba: 7 conexões
  Kaye Donac

In [9]:
# Visualize basic graph statistics
import networkx as nx
import collections

G = nx.read_graphml("data/processed/kg/fashion_kg.graphml")

print(f"Knowledge Graph Statistics")
print(f"  Nodes (entities):  {G.number_of_nodes()}")
print(f"  Edges (relations): {G.number_of_edges()}")

# Relation type distribution
relation_counts = collections.Counter(
    data["relation"] for _, _, data in G.edges(data=True)
)
print(f"\nRelation type distribution:")
for rel, count in relation_counts.most_common():
    print(f"  {rel}: {count}")

Knowledge Graph Statistics
  Nodes (entities):  197
  Edges (relations): 194

Relation type distribution:
  USES_MATERIAL: 35
  ASSOCIATED_WITH: 35
  CO_OCCURS_WITH: 33
  FEATURES_SILHOUETTE: 29
  INSPIRED_BY: 16
  USES_COLOR: 14
  REFERENCES_CULTURE: 14
  DESIGNED_BY: 10
  PRESENTED_AT: 8


---
## Step 4 — Run Comparative Experiment

Runs the three evaluation questions under both conditions.

**Condition A (LLM-only):** raw text → LLM → response  
**Condition B (KG+LLM):** triples subgraph → LLM → response

**Questions:**
- P1 (intra-document): What are the dominant trends in this collection?
- P2 (intra-document): What relations between elements characterize these trends?
- P3 (inter-document): What trends dominated London AW25?

In [10]:
!python src/04-query.py

Grafo carregado: 197 nós, 194 arestas

Reviews encontrados: 9


Review: 2025-lfw-burberry-review.txt
Triplas no subgrafo: 23
  [P1] What are the dominant trends in this collection?
    → Condição A salva
    → Condição B salva
  [P2] What relations between elements characterize these trends?
    → Condição A salva
    → Condição B salva

Review: 2025-lfw-di-petsa-review.txt
Triplas no subgrafo: 19
  [P1] What are the dominant trends in this collection?
    → Condição A salva
    → Condição B salva
  [P2] What relations between elements characterize these trends?
    → Condição A salva
    → Condição B salva

Review: 2025-lfw-dilara-findikoglu-review.txt
Triplas no subgrafo: 24
  [P1] What are the dominant trends in this collection?
    → Condição A salva
    → Condição B salva
  [P2] What relations between elements characterize these trends?
    → Condição A salva
    → Condição B salva

Review: 2025-lfw-erdem-review.txt
Triplas no subgrafo: 21
  [P1] What are the dominant trends in th

---
## Step 5 — Inspect Results

Compare responses from both conditions side by side.

In [11]:
# Compare Condition A vs B for Erdem P1
def read_result(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

brand = "2025-lfw-erdem-review"
question = "P1"

result_a = read_result(f"results/condition_a/{brand}_{question}.txt")
result_b = read_result(f"results/condition_b/{brand}_{question}.txt")

print("=" * 60)
print("CONDITION A — LLM-only")
print("=" * 60)
print(result_a)

print()
print("=" * 60)
print("CONDITION B — KG+LLM")
print("=" * 60)
print(result_b)

CONDITION A — LLM-only
FILE: 2025-lfw-erdem-review
QUESTION: P1
CONDITION: LLM-only

The dominant trends in the Erdem Fall 2025 collection include:

1. **Fluidity and Ethereality**: The collection features more fluid silhouettes, moving away from strict "cookie cutter" shapes to embrace a softer, dreamlike quality, as seen in the use of organza and the construction of dresses that billow behind the models.

2. **Artistic Collaboration**: The integration of contemporary art, specifically the work of Scottish painter Kaye Donachie, is a significant trend. The collection reflects her ethereal renderings and painterly techniques, translating them into fabric and fashion.

3. **Innovative Fabric Techniques**: There is a focus on technical craftsmanship, utilizing simple fabrics like cotton and neoprene, and employing methods such as laser-cutting, embroidery, and bonding to create unique textures and silhouettes.

4. **Emotive and Poetic Themes**: The collection explores themes of emotion a

In [12]:
# Compare Condition A vs B for P3 (inter-document)
result_a_p3 = read_result("results/condition_a/corpus_lfw_aw25_P3.txt")
result_b_p3 = read_result("results/condition_b/corpus_lfw_aw25_P3.txt")

print("=" * 60)
print("P3 — CONDITION A — LLM-only")
print("=" * 60)
print(result_a_p3)

print()
print("=" * 60)
print("P3 — CONDITION B — KG+LLM")
print("=" * 60)
print(result_b_p3)

P3 — CONDITION A — LLM-only
FILE: corpus_lfw_aw25
QUESTION: P3
CONDITION: LLM-only

The trends that dominated London Autumn/Winter 2025 (AW25) included:

1. **Countryside Inspiration**: Burberry's collection showcased a pastoral theme with a focus on British countryside aesthetics, featuring a color palette of browns, greens, and burgundies, along with classic outerwear like trenches and bombers.

2. **Sculptural and Textural Designs**: Roksanda's collection emphasized sculptural forms and rich textures, inspired by a woman sculptor, blending everyday materials with sophisticated designs.

3. **Exploration of Feminine Identity**: Dilara Findikoglu's collection focused on themes of chaos and control, empowering women to define their own narratives, while Simone Rocha's work highlighted a slow evolution of hyper-feminine styles.

4. **Craft Techniques and Artistic References**: S.S. Daley drew inspiration from the Scottish Colourists, incorporating craft techniques into wearable art, whi

## Step 6 — Save Results

Downloads all experiment results as a zip file for upload to the repository.

In [13]:
from google.colab import files
import shutil

shutil.make_archive("results", "zip", "results")
files.download("results.zip")
print("results.zip downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

results.zip downloaded.


---
## Conclusion

All results presented in this notebook can be reproduced by anyone with access to:
1. This repository: https://github.com/diegofelixlima/beyond-frequency
2. An OpenAI API key
3. Google Colab (free tier is sufficient)

The notebook contains data, code, and results in a single versioned document,  
in accordance with Open Science principles.

**Evaluation of results** (interpretability and groundedness) is documented  
in the article: `doc/article.tex`